# 🎯 Workshop Day 3: Evaluation & Optimization

**Agentic RAG Workshop** | Day 3 of 3

---

### 🎯 Translated English text
1. **Translated English text RAG** — Translated English text RAGAS (4 metrics)
2. **Translated English text Eval Dataset** — Translated English text Gemini
3. **Translated English text Agent** — Tool Selection, LLM-as-Judge
4. **Translated English text Pipeline** — A/B Experiment
5. **Prompt Engineering** — Translated English text Instruction Translated English text
6. **Capstone Project** — Translated English text 3 Translated English text ⭐

### 🗺️ Translated English text 3 Translated English text

```
Day 1 (Data)           Day 2 (Agent)          Day 3 (Evaluation)
─────────────          ─────────────          ─────────────────
Raw → Chunk →          Agent + Tool →         Translated English text (3.1-3.2)
Embed → VectorDB       RAG Agent →            Translated English text Agent (3.3)
  → Retrieve           Multi-Agent →          Translated English text (3.4-3.5)
                       Agentic RAG            Capstone ⭐ (3.6)
```

> 💡 Translated English text "Translated English text" — Translated English text

## 📦 Section 0: Translated English text Dependencies

In [ ]:
%%time
pip3 install python-pptx 2>&1 | tail -3 install -q ragas datasets google-adk google-genai \
    sentence-transformers qdrant-client langchain-text-splitters \
    rank_bm25 scikit-learn langchain-google-genai langchain-huggingface

import warnings
warnings.filterwarnings('ignore')
print('✅ Translated English text!')

In [ ]:
!pip install -q langchain-huggingface

In [ ]:
%%time
import os, json, hashlib, random, asyncio
import numpy as np

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ Translated English text API Key Translated English text Colab Secrets')
except Exception:
    api_key = input('🔑 Translated English text Gemini API Key: ')
    os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

from google import genai
try:
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    resp = client.models.generate_content(model='gemini-2.5-flash', contents='Translated English text OK')
    print(f'✅ API Translated English text: {(resp.text.strip() if resp.text else '(Translated English text output)')}')
except Exception as e:
    print(f'❌ API Error: {e}')

### Translated English text + VectorDB (re-use Translated English text Day 1-2)

In [ ]:
%%time
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')

documents = {
    'healthcare': 'Translated English text AI Translated English text X-ray Translated English text 30 Translated English text 5 Translated English text Deep Learning Translated English text 95% Translated English text',
    'banking': 'Translated English text KADE AI Assistant Translated English text RAG Translated English text 5 Translated English text 30 Translated English textThai languageTranslated English text',
    'education': 'Translated English text RAG Translated English text-Translated English text PDF Translated English text 500 Translated English text vector embeddings Translated English text multilingual model Translated English textThai language studentsTranslated English text',
    'agriculture': 'Translated English text AI Translated English text 92% Translated English text 40% Translated English text Computer Vision Translated English text CNN Translated English text train Translated English text 50000 Translated English text',
    'rag': 'Translated English text RAG Translated English text 3 Translated English text Retriever Translated English text VectorDB Reader Translated English text Generator Translated English text hallucination Translated English text LLM Translated English text',
    'embedding': 'Text Embedding Translated English text vector multilingual-e5-large Translated English text 100 Translated English textThai language Translated English text vector 1024 Translated English text vector Translated English text Cosine Similarity Translated English text',
    'kmitl': 'Translated English text (KMITL) Translated English text AI Translated English text Smart Campus Translated English text IoT sensor Translated English text Machine Learning Translated English text 25% Translated English text NLP Thai languageTranslated English textstudents Translated English text RAG Translated English text AI Tutor Translated English text',
    'nlp': 'NLP Thai languageTranslated English text Thai languageTranslated English text Word Segmentation Translated English text PyThaiNLP Translated English text Tokenizer Translated English text',
    'vectordb': 'Qdrant Translated English text Vector Database Translated English text ANN Translated English text vectors Translated English text Cosine Dot Product L2 Distance Translated English text payload filter Translated English text metadata Translated English text',
}

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
all_chunks, all_sources = [], []
for src, text in documents.items():
    for chunk in splitter.split_text(text):
        all_chunks.append(chunk)
        all_sources.append(src)

passages = [f'passage: {c}' for c in all_chunks]
embeddings = embed_model.encode(passages, show_progress_bar=True)
VECTOR_SIZE = embeddings.shape[1]

qdrant = QdrantClient(':memory:')
qdrant.create_collection('thai_ai_kb',
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE))
points = [models.PointStruct(id=i, vector=embeddings[i].tolist(),
    payload={'text': all_chunks[i], 'source': all_sources[i]}) for i in range(len(all_chunks))]
qdrant.upsert('thai_ai_kb', points=points)

print(f'✅ Data ready: {len(all_chunks)} chunks, {len(documents)} sources')

# RAG function
def search_qdrant(query, top_k=3):
    q_vec = embed_model.encode(f'query: {query}').tolist()
    results = qdrant.query_points('thai_ai_kb', query=q_vec, limit=top_k).points
    return [{'text': r.payload['text'], 'source': r.payload['source'],
             'score': round(r.score, 4)} for r in results]

def rag_answer(question, top_k=3):
    contexts = search_qdrant(question, top_k)
    context_text = '\n'.join([c['text'] for c in contexts])
    prompt = f'"""Translated English textThai language Translated English text\n\nTranslated English text:\n{context_text}\n\nTranslated English text: {question}\nTranslated English text:"""'
    resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.3))
    return (resp.text.strip() if resp.text else '(Translated English text output)'), contexts

# Quick test
ans, ctx = rag_answer('AI Translated English text')
print(f'\n🧪 Test RAG: {ans[:80]}...')

---
## 📊 Section 3.1: RAGAS — Translated English text RAG

### RAGAS Translated English text?

**RAGAS (Retrieval-Augmented Generation Assessment Suite)** = framework Translated English text RAG

```
Input: question + answer + contexts (+ ground_truth)
  ↓ RAGAS Translated English text
Output: points 0.0 - 1.0 Translated English text metric
```

### 4 Metrics Translated English text

| Metric | Translated English text | Translated English text | Translated English text ground_truth? |
|--------|---------|------------|:------------------:|
| **Faithfulness** | Translated English text "Translated English text" Translated English text? (Translated English text?) | answer ↔ contexts | ❌ |
| **Answer Relevancy** | Translated English text? | answer ↔ question | ❌ |
| **Context Precision** | Translated English text chunk Translated English text? | contexts ↔ ground_truth | ✅ |
| **Context Recall** | Translated English text chunk Translated English text? | contexts ↔ ground_truth | ✅ |

### 🌡️ Translated English textpoints

| points | Translated English text | Translated English text |
|:-----:|-------|----------|
| 0.8-1.0 | 🟢 Translated English text | ✅ Translated English text |
| 0.6-0.8 | 🟡 Translated English text | ⚠️ Translated English text |
| < 0.6 | 🔴 Translated English text | ❌ Translated English text |

In [ ]:
%%time
# ─── Translated English text Evaluation Dataset ───
# Translated English text: question, answer, contexts, ground_truth

eval_questions = [
    'AI Translated English text',
    'RAG Translated English text',
    'Translated English text AI Translated English text',
    'NLP Thai languageTranslated English text',
    'Qdrant Translated English text',
]

eval_ground_truths = [
    'Translated English text AI Translated English text X-ray Translated English text 30 Translated English text 5 Translated English text 95%',
    'RAG Translated English text 3 Translated English text Retriever Translated English text VectorDB, Reader Translated English text, Generator Translated English text hallucination',
    'Translated English text KADE AI Assistant Translated English text RAG Translated English text 5 Translated English text 30 Translated English text',
    'Thai languageTranslated English text Word Segmentation Translated English text PyThaiNLP Translated English text',
    'Qdrant Translated English text Vector Database Translated English text ANN Translated English text Cosine Dot Product L2',
]

# Translated English text RAG Translated English text answers + contexts
eval_answers = []
eval_contexts = []

print('🔄 Translated English text answers...')
for q in eval_questions:
    ans, ctxs = rag_answer(q)
    eval_answers.append(ans)
    eval_contexts.append([c['text'] for c in ctxs])
    print(f'  ✅ {q[:30]}... → {ans[:40]}...')

print(f'\n📊 Eval Dataset: {len(eval_questions)} questions')

In [ ]:
%%time
# ─── Translated English text RAGAS (Translated English text Gemini LLM + Local Embeddings) ───
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd

# 1. Translated English text Gemini Translated English text LLM Translated English text (Reasoning)
gemini_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=os.environ['GOOGLE_API_KEY'],
))

# 2. Translated English text Local Embedding (E5) Translated English text API 404
local_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-large')
)

# 3. Translated English text Dataset
eval_dataset = Dataset.from_dict({
    'question': eval_questions,
    'answer': eval_answers,
    'contexts': eval_contexts,
    'ground_truth': eval_ground_truths,
})

# 4. Translated English text RAGAS
try:
    result = evaluate(
        dataset=eval_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=gemini_llm,
        embeddings=local_embeddings,
    )

    # Translated English text: Translated English text result Translated English text result.scores
    final_data = []
    if hasattr(result, 'scores'):
        # Translated English text scores Translated English text list Translated English text to_pandas Translated English text
        if isinstance(result.scores, list):
            final_data = result.scores
        else:
            final_data = result.scores.to_pandas()
    elif isinstance(result, list):
        final_data = result
    else:
        final_data = [result]

    scores_df = pd.DataFrame(final_data)

    # Translated English text Metric
    avg_scores = scores_df.mean(numeric_only=True)
    for metric, score in avg_scores.items():
        level = '🟢' if score >= 0.8 else ('🟡' if score >= 0.6 else '🔴')
        print(f'  {level} {metric}: {score:.4f}')

except Exception as e:
    print(f'⚠️ RAGAS error: {e}')
    print('💡 Translated English text error Translated English text Section 3.3 (LLM-as-Judge) Translated English text')

### 💡 Observation
- **Faithfulness Translated English text** = Agent Translated English text
- **Context Precision Translated English text** = Translated English text chunk Translated English text → Translated English text `chunk_size`, `top_k`
- **Context Recall Translated English text** = Translated English text → Translated English text `top_k` Translated English text embedding

> 🎯 Translated English text: Translated English text metric **≥ 0.7** Translated English text production

### 🧪 Translated English text 3.1
1. Translated English text 3 Translated English text + ground_truth
2. Translated English text RAGAS → metric Translated English text?
3. Translated English text

In [ ]:
# 🧪 Translated English text: Translated English text RAGAS
# 💡 Hint:
#   1. Translated English text question + ground_truth Translated English text list
#   2. Translated English text rag_answer() Translated English text answer + contexts
#   3. Translated English text Dataset.from_dict() Translated English text evaluate()


---
## 🔬 Section 3.2: Translated English text Evaluation Dataset Translated English text

### Translated English text?

- Translated English text ground_truth Translated English text → Translated English text (5 Translated English text = 30 Translated English text)
- Translated English text LLM Translated English text chunk → Translated English text 50 Translated English text 2 Translated English text

```
Chunk: "Translated English text AI Translated English text X-ray Translated English text 95%"
  ↓ Gemini Translated English text
Q: "Translated English text AI Translated English text X-ray?"
A: "Translated English text 95%"
```

In [ ]:
%%time
# ─── Auto-generate Q&A Translated English text chunks ───
def generate_qa_from_chunk(chunk_text):
    prompt = f'''Translated English text 1 Translated English text + Translated English text
Translated English text: {chunk_text}

Translated English text JSON: {{"question": "...", "answer": "..."}}
Translated English textThai language Translated English text'''
    try:
        resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
            config=genai.types.GenerateContentConfig(temperature=0.3, response_mime_type='application/json'))
        return json.loads(resp.text)
    except:
        return None

# Translated English text 8 chunks (Translated English text chunk Translated English text source)
auto_qa = []
seen_sources = set()
for i, chunk in enumerate(all_chunks):
    src = all_sources[i]
    if src in seen_sources: continue
    seen_sources.add(src)
    qa = generate_qa_from_chunk(chunk)
    if qa:
        qa['source'] = src
        auto_qa.append(qa)
        print(f'  ✅ [{src}] Q: {qa["question"][:50]}...')

print(f'\n📊 Translated English text {len(auto_qa)} Q&A pairs')

### 💡 Observation
- LLM Translated English text 10 Translated English text
- Translated English text **Translated English text** — Translated English text
- Translated English text **starting point** Translated English text

> 🎯 Production: Translated English text 100 Translated English text → Translated English text → Translated English text 50 Translated English text

### 🧪 Translated English text 3.2
Translated English text auto Q&A Translated English text chunks Translated English text

In [ ]:
# 🧪 Translated English text
# 💡 Hint: Translated English text all_chunks Translated English text Q&A Translated English text generate_qa_from_chunk()


---
## 🧪 Section 3.3: Translated English text Agent

### Translated English text?

| Translated English text | Translated English text | Translated English text |
|--------|----------|--------|
| **Tool Selection** | Translated English text tool Translated English text? | Translated English text BMI → Translated English text calculate_bmi |
| **Response Quality** | Translated English text? | Translated English text LLM-as-Judge |
| **Edge Cases** | Translated English text? | Translated English text, Translated English text |
| **Guardrails** | Translated English text? | Translated English text |

In [ ]:
# ─── Translated English text Agent Translated English text ───
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

def search_knowledge(query: str) -> dict:
    """Translated English text AI Translated English text
    Args:
        query: Translated English text
    """
    results = search_qdrant(query, top_k=3)
    return {'status': 'success', 'results': results}

def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """Translated English text BMI
    Args:
        weight_kg: Translated English text
        height_cm: Translated English text
    """
    h = height_cm / 100
    bmi = weight_kg / (h ** 2)
    return {'bmi': round(bmi, 1)}

test_agent = Agent(
    name='test_agent', model='gemini-2.5-flash',
    instruction='''Translated English text AI Translated English textThai language
    - Translated English text search_knowledge Translated English text AI Translated English text
    - Translated English text calculate_bmi Translated English text BMI
    - Translated English text
    - Translated English text''',
    tools=[search_knowledge, calculate_bmi]
)

async def test_agent_call(agent, msg):
    runner = InMemoryRunner(agent=agent, app_name=agent.name)
    session_id = f's_{id(agent)}_{id(msg)}'
    await runner.session_service.create_session(
        app_name=agent.name, user_id='tester', session_id=session_id
    )
    content = types.Content(role='user', parts=[types.Part.from_text(text=msg)])
    response, tools = '', []
    async for event in runner.run_async(user_id='tester', session_id=session_id, new_message=content):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text: response = p.text
                if p.function_call: tools.append(p.function_call.name)
    return response, tools

print('✅ Test Agent Translated English text')

In [ ]:
# ─── Test Suite: Tool Selection ───
test_cases = [
    {'input': 'Translated English text 70 Translated English text 175 BMI?',      'expected_tool': 'calculate_bmi'},
    {'input': 'AI Translated English text',            'expected_tool': 'search_knowledge'},
    {'input': 'Translated English text',                     'expected_tool': None},
    {'input': 'Qdrant Translated English text',                 'expected_tool': 'search_knowledge'},
    {'input': 'Translated English text',            'expected_tool': None},
]

print('═' * 60)
print('🧪 Test Suite: Tool Selection')
print('═' * 60)

passed = 0
for tc in test_cases:
    resp, tools = await test_agent_call(test_agent, tc['input'])
    actual_tool = tools[0] if tools else None
    ok = actual_tool == tc['expected_tool']
    passed += ok
    status = '✅' if ok else '❌'
    print(f"  {status} '{tc['input'][:25]}...' → expected={tc['expected_tool']}, got={actual_tool}")

print(f'\n📊 Pass rate: {passed}/{len(test_cases)} ({passed/len(test_cases)*100:.0f}%)')

In [ ]:
%%time
# ─── LLM-as-Judge (Translated English text) ───
def llm_judge(question, answer, max_score=5):
    prompt = f'''Translated English textpointsTranslated English text 1-{max_score} Translated English text (Translated English textpointsTranslated English text):
- Translated English text: Translated English text (Translated English text 2 points)
- Translated English text: Translated English text (Translated English text = Translated English text 1 points)
- Translated English text: Translated English text bullet points Translated English text 5 Translated English text (Translated English text = Translated English text 1 points)
- Translated English text: Translated English text
- Translated English text: Translated English text "Translated English text" (Translated English text = Translated English text 1 points)

Translated English text: {question}
Translated English text: {answer}

Translated English text JSON: {{"score": 1-{max_score}, "reason": "Translated English text"}}'''
    try:
        resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
            config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
        return json.loads(resp.text)
    except:
        return {'score': 0, 'reason': 'error'}

# Translated English text — Translated English text (Translated English text + Translated English text + edge case)
print('═' * 60)
print('🧪 LLM-as-Judge: Translated English text')
print('═' * 60)

judge_questions = [
    'AI Translated English text',
    'Translated English text AI Translated English text vs Translated English text',
    'Translated English text AI Translated English text IPO Translated English text',
]

total_score = 0
for q in judge_questions:
    ans, _ = rag_answer(q)
    verdict = llm_judge(q, ans)
    total_score += verdict['score']
    stars = '⭐'*verdict['score'] + '☆'*(5-verdict['score'])
    print(f'  {stars} {verdict["score"]}/5 | {q[:30]}... → {verdict["reason"]}')

avg = total_score / len(judge_questions)
print(f'\n📊 Average: {avg:.1f}/5.0')


### 💡 Observation
- **Tool Selection test** = Translated English text Agent Translated English text tool Translated English text (deterministic)
- **LLM-as-Judge** = Translated English text LLM Translated English text (Translated English text)
- Translated English text → Translated English text "Translated English text" Translated English text "Translated English text"

> ⚠️ LLM-as-Judge Translated English text 100% Translated English text — Translated English text screening tool Translated English text human review

### 🧪 Translated English text 3.3
Translated English text test cases 5 Translated English text edge cases (Translated English text, Translated English text)

In [ ]:
# 🧪 Translated English text
# 💡 Hint: Translated English text test_cases Translated English text test_agent_call()


---
## ⚡ Section 3.4: Translated English text RAG Pipeline

### Translated English text?

| Translated English text | Parameter | Translated English text |
|---------|----------|----------|
| Chunking | `chunk_size`, `overlap` | Context Precision |
| Retrieval | `top_k`, `alpha` | Context Recall |
| Generation | `temperature`, prompt | Faithfulness, Relevancy |

### A/B Experiment
```
Config A: chunk=100, top_k=3
Config B: chunk=200, top_k=5  ← Translated English text?
Config C: chunk=300, top_k=3
```

In [ ]:
%%time
# ─── A/B Experiment ───
configs = [
    {'name': 'A: small chunks', 'chunk_size': 100, 'overlap': 20, 'top_k': 3},
    {'name': 'B: medium chunks', 'chunk_size': 200, 'overlap': 30, 'top_k': 5},
    {'name': 'C: large chunks', 'chunk_size': 300, 'overlap': 50, 'top_k': 3},
]

test_q = 'AI Translated English text'
test_gt = 'Translated English text AI Translated English text X-ray Translated English text 95%'

print('═' * 60)
print('🧪 A/B Experiment: Translated English text Config')
print('═' * 60)

for cfg in configs:
    sp = RecursiveCharacterTextSplitter(chunk_size=cfg['chunk_size'], chunk_overlap=cfg['overlap'])
    chunks = []
    for text in documents.values():
        chunks.extend(sp.split_text(text))
    
    # Embed + search
    vecs = embed_model.encode([f'passage: {c}' for c in chunks])
    q_vec = embed_model.encode(f'query: {test_q}')
    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity([q_vec], vecs)[0]
    top_idx = np.argsort(sims)[-cfg['top_k']:][::-1]
    top_chunks = [chunks[i] for i in top_idx]
    top_scores = [sims[i] for i in top_idx]
    
    # Check if ground truth info is in retrieved chunks
    gt_found = any('Translated English text' in c or '95%' in c for c in top_chunks)
    
    print(f'\n📋 {cfg["name"]}: {len(chunks)} chunks')
    print(f'   Top scores: {[round(s,3) for s in top_scores]}')
    print(f'   Ground truth found: {"✅" if gt_found else "❌"}')
    print(f'   Best chunk: {top_chunks[0][:60]}...')

### 💡 Observation
- **chunk Translated English text** (100) → chunks Translated English text
- **chunk Translated English text** (300) → Translated English text noise Translated English text
- **top_k Translated English text** → recall Translated English text precision Translated English text

> 🎯 Translated English text config Translated English text — Translated English text **Translated English text** Translated English text

### 🧪 Translated English text 3.4
Translated English text 3 configs → Translated English text config Translated English text

In [ ]:
# 🧪 Translated English text: Translated English text configs Translated English text
# 💡 Hint: Translated English text configs Translated English text list Translated English text


---
## ✍️ Section 3.5: Prompt Engineering Translated English text Agent

### Translated English text

| Translated English text | Translated English text | Translated English text Hallucination? |
|--------|---------|:-----------------:|
| **Role** | "Translated English text..." | ⭐ |
| **Few-shot** | Translated English text Q&A | ⭐⭐ |
| **Chain-of-Thought** | "Translated English text..." | ⭐⭐⭐ |
| **Guardrails** | "Translated English text" | ⭐⭐⭐ |
| **Output Format** | "Translated English text bullet 3 Translated English text" | ⭐ |

In [ ]:
# ─── Before vs After Prompt ───
prompt_before = 'Translated English text'
prompt_after = '''Translated English text AI Translated English textThai language

Translated English text:
1. Translated English text search_knowledge Translated English text
2. Translated English text
3. Translated English text [healthcare] [banking]
4. Translated English text "Translated English text"
5. Translated English text bullet points Translated English text 5 Translated English text'''

agent_v1 = Agent(name='v1', model='gemini-2.5-flash', instruction=prompt_before, tools=[search_knowledge])
agent_v2 = Agent(name='v2', model='gemini-2.5-flash', instruction=prompt_after, tools=[search_knowledge])

# Translated English text — Translated English text instruction Translated English text
hard_test_questions = [
    'Translated English text AI Translated English text vs Translated English text',
    'Translated English text AI Translated English text',
    'Translated English text NLP Thai languageTranslated English text',
]

print('═' * 60)
print('🧪 Before vs After Prompt (Translated English text)')
print('═' * 60)

v1_total, v2_total = 0, 0
for q in hard_test_questions:
    r1, _ = await test_agent_call(agent_v1, q)
    r2, _ = await test_agent_call(agent_v2, q)

    j1 = llm_judge(q, r1)
    j2 = llm_judge(q, r2)
    v1_total += j1['score']
    v2_total += j2['score']

    print(f'\n❓ {q}')
    print(f'  V1: {j1["score"]}/5 — {j1["reason"]}')
    print(f'  V2: {j2["score"]}/5 — {j2["reason"]}')

v1_avg = v1_total / len(hard_test_questions)
v2_avg = v2_total / len(hard_test_questions)
print(f'\n{"═"*60}')
print(f'📊 Translated English text: V1 avg = {v1_avg:.1f}/5  |  V2 avg = {v2_avg:.1f}/5')
diff = v2_avg - v1_avg
if diff > 0:
    print(f'📈 V2 Translated English text V1 +{diff:.1f} points — instruction Translated English text!')
elif diff < 0:
    print(f'📉 V1 Translated English text V2 — Translated English text instruction Translated English text')
else:
    print(f'➡️ Translated English text — Translated English text')


### 💡 Observation
- Instruction Translated English text → Translated English text **Translated English text, Translated English text, Translated English text**
- Instruction Translated English text → Translated English text **Translated English text, Translated English text, Translated English text reference**
- **Guardrails** ("Translated English text") Translated English text hallucination Translated English text

> 🎯 Prompt Engineering = Translated English text **Translated English text**

### 🧪 Translated English text 3.5
Translated English text instruction 3 Translated English text → Translated English text LLM-as-Judge → Translated English text

In [ ]:
# 🧪 Translated English text
# 💡 Hint: Translated English text agent_v3 Translated English text instruction Translated English text


---
## 🚀 Section 3.6: Capstone Project ⭐

### Translated English text 3 Translated English text

```
Day 1: Translated English text → Chunk → Embed → VectorDB
Day 2: Translated English text Agent → Tools → RAG Agent → Multi-Agent
Day 3: Translated English text RAGAS → Translated English text → Translated English text → Translated English text
```

### Translated English text
1. **Translated English text RAG Pipeline** Translated English text
2. **Translated English text** Translated English text RAGAS + LLM-as-Judge
3. **Translated English text** Translated English text (chunk_size, prompt, top_k)
4. **Translated English text Before/After**

In [ ]:
# ─── Capstone: Before → After ───
print('═' * 60)
print('🚀 Capstone: Translated English text RAG Pipeline')
print('═' * 60)

# Translated English text (Translated English text + Translated English text + edge case) Translated English text
capstone_questions = [
    'AI Translated English text',
    'Translated English text embedding model Translated English textThai languageTranslated English text',
    'Translated English text-Translated English text RAG Translated English text fine-tuning',
    'Elon Musk Translated English text',
    'Translated English text use case Translated English text AI Translated English text',
]

# Step 1: Translated English text baseline (rag_answer Translated English text)
print('\n📊 Step 1: Baseline (RAG Translated English text)')
baseline_scores = []
for q in capstone_questions:
    ans, _ = rag_answer(q)
    verdict = llm_judge(q, ans)
    baseline_scores.append(verdict['score'])
    stars = '⭐'*verdict['score'] + '☆'*(5-verdict['score'])
    print(f'  {stars} {verdict["score"]}/5 | {q[:35]}...')
baseline_avg = sum(baseline_scores) / len(baseline_scores)
print(f'   → Baseline Average: {baseline_avg:.1f}/5.0')

# Step 2: Translated English text (Translated English text Agent + instruction Translated English text)
print('\n📊 Step 2: After Optimization (Agent + Good Prompt)')
improved_scores = []
for q in capstone_questions:
    r, _ = await test_agent_call(agent_v2, q)
    verdict = llm_judge(q, r)
    improved_scores.append(verdict['score'])
    stars = '⭐'*verdict['score'] + '☆'*(5-verdict['score'])
    print(f'  {stars} {verdict["score"]}/5 | {q[:35]}...')
improved_avg = sum(improved_scores) / len(improved_scores)
print(f'   → Improved Average: {improved_avg:.1f}/5.0')

# Step 3: Translated English text
print(f'\n{"═"*60}')
print(f'📊 Translated English text Before → After')
print(f'{"═"*60}')
print(f'  Baseline:  {"⭐" * round(baseline_avg)} {baseline_avg:.1f}/5.0')
print(f'  Improved:  {"⭐" * round(improved_avg)} {improved_avg:.1f}/5.0')
diff = improved_avg - baseline_avg
if diff > 0:
    print(f'  📈 +{diff:.1f} Translated English text!')
elif diff < 0:
    print(f'  📉 {diff:.1f} Translated English text — Translated English text baseline')
else:
    print(f'  ➡️ Translated English text — Translated English text')


### 💡 Observation
- **Prompt Engineering** Translated English text
- **Chunk size** Translated English text — Translated English text
- **Translated English text-Translated English text** Translated English text — Translated English text

> 🎯 "What gets measured gets improved" — Translated English text

### 🧪 Translated English text 3.6
1. Translated English text 2 Translated English text (prompt + chunk_size Translated English text top_k)
2. Translated English text Before/After
3. Translated English text

In [ ]:
# 🧪 Capstone: Translated English text
# 💡 Hint:
#   1. Translated English text chunk_size → re-embed → re-search → Translated English text
#   2. Translated English text instruction → Translated English text agent Translated English text → Translated English text
#   3. Translated English text top_k → rag_answer( ,top_k=5) → Translated English text


---
## 🎯 Translated English text: Translated English text 3 Translated English text

| Day | content | Translated English text |
|-----|--------|-------------|
| **Day 1** | Data Engineering Pipeline | Qdrant, E5-large, BM25 |
| **Day 2** | Agent Building | Google ADK, Tools, Multi-Agent |
| **Day 3** | Evaluation & Optimization | RAGAS, LLM-as-Judge |

### Skills Translated English text
- ✅ Translated English text RAG Pipeline Translated English text
- ✅ Translated English text Agent Translated English text
- ✅ Translated English text + Translated English text
- ✅ Translated English textThai languageTranslated English text

### 🛣️ Translated English text?
- **Production**: Deploy Translated English text Cloud (GCP, AWS) + Database session
- **Advanced**: Fine-tune embedding Translated English text domain Translated English text
- **Scale**: Qdrant Cloud + horizontal scaling
- **Monitor**: Log + alert Translated English text

---

**🙏 Translated English text 3 Translated English text!**

> "The best way to learn AI is to build with AI."